# BOA Constrictor — Alternative Backbone Experiment
**GSoC 2025 Evaluation Task — Part 3**

This notebook:
1. Forks + clones the BOA repo
2. Installs all dependencies
3. Patches the codebase to support LSTM and Transformer backbones
4. Runs compression with each backbone
5. Benchmarks and compares throughput vs compression ratio

## Step 0 — Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU found — switch to GPU runtime')

Thu Mar 12 21:25:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Step 1 — Clone the fork

In [ ]:
import os

# Clone the fork
!git clone https://github.com/electricdystopia/boa-constrictor.git
os.chdir('boa-constrictor')
!ls -la

Cloning into 'boa-constrictor'...
remote: Enumerating objects: 340, done.
remote: Counting objects: 100% (111/111), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 340 (delta 71), reused 67 (delta 58), pack-reused 229 (from 1)
Receiving objects: 100% (340/340), 104.18 MiB | 35.58 MiB/s, done.
Resolving deltas: 100% (123/123), done.
total 232
drwxr-xr-x 5 root root  4096 Mar 12 22:10 .
drwxr-xr-x 7 root root  4096 Mar 12 22:10 ..
-rw-r--r-- 1 root root 17894 Mar 12 22:10 alternative_backbones.py
-rw-r--r-- 1 root root 14221 Mar 12 22:10 boa.py
-rw-r--r-- 1 root root 12361 Mar 12 22:10 codec.py
-rw-r--r-- 1 root root 22116 Mar 12 22:10 evaluator.py
drwxr-xr-x 8 root root  4096 Mar 12 22:10 experiments
drwxr-xr-x 8 root root  4096 Mar 12 22:10 .git
-rw-r--r-- 1 root root  1202 Mar 12 22:10 .gitignore
-rw-r--r-- 1 root root 25981 Mar 12 22:10 gpu_range_coder.py
-rw-r--r-- 1 root root 34523 Mar 12 22:10 LICENSE
-rw-r--r-- 1 root root 29509 Mar 12 22:10 main.py
-rw-r--r-

## Step 2 — Install dependencies

In [ ]:
!pip install https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.1.post4/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
!pip install https://github.com/state-spaces/mamba/releases/download/v2.3.1/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.6/287.6 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 533.6/533.6 MB 1.1 MB/s eta 0:00:00


In [ ]:
!pip install mambapy
!pip install pybind11
!pip install uproot awkward
!pip install constriction

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.2/310.2 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.4/395.4 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 919.6/919.6 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 656.7/656.7 kB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 412.1/412.1 kB 8.2 MB/s eta 0:00:00


## Step 3 — Smoke test: verify backbones run correctly

In [ ]:
import torch
from alternative_backbones import build_boa_model

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

for backbone in ['lstm', 'transformer']:
    model = build_boa_model(backbone=backbone, d_model=128, num_layers=2,
                            max_seq_len=256).to(device)
    x = torch.randint(0, 256, (2, 64), device=device)
    logits, state = model(x)
    assert logits.shape == (2, 64, 256)
    print(f'  {backbone:>12}: OK — output {logits.shape}, params={model.count_parameters():,}')

Using device: cuda
          lstm: OK — output torch.Size([2, 64, 256]), params=329,984
   transformer: OK — output torch.Size([2, 64, 256]), params=494,336


## Step 4 — Throughput benchmark

In [ ]:
import time
import pandas as pd

def benchmark_backbone(backbone, d_model=256, num_layers=4,
                        seq_len=1024, batch_size=4, n_iters=30, max_seq_len=2048):
    model = build_boa_model(backbone=backbone, d_model=d_model,
                            num_layers=num_layers, max_seq_len=max_seq_len).to(device)
    model.eval()
    x = torch.randint(0, 256, (batch_size, seq_len), device=device)
    total_bytes = batch_size * seq_len

    # Warm-up
    with torch.no_grad():
        for _ in range(5): model(x)

    start = time.perf_counter()
    with torch.no_grad():
        for _ in range(n_iters): model(x)
    elapsed = time.perf_counter() - start

    return {
        'backbone': backbone,
        'parameters': model.count_parameters(),
        'throughput_bytes_per_sec': (total_bytes * n_iters) / elapsed,
        'latency_ms_per_batch': (elapsed / n_iters) * 1000,
        'seq_len': seq_len,
        'device': device,
    }

results = []
for b in ['lstm', 'transformer']:
    r = benchmark_backbone(b, d_model=256, num_layers=4, seq_len=1024,
                           batch_size=4, max_seq_len=2048)
    results.append(r)
    print(f"{b:>12}: {r['throughput_bytes_per_sec']:>12,.0f} B/s  "
          f"{r['latency_ms_per_batch']:>8.2f} ms/batch  "
          f"params={r['parameters']:,}")

df = pd.DataFrame(results)
print('\nResults DataFrame:')
print(df.to_string(index=False))

        lstm:       83,327 B/s     49.16 ms/batch  params=2,236,672
 transformer:      220,899 B/s     18.54 ms/batch  params=3,811,072

Results DataFrame:
   backbone  parameters  throughput_bytes_per_sec  latency_ms_per_batch  seq_len device
       lstm     2236672              83326.660408             49.155936     1024   cuda
transformer     3811072             220898.611850             18.542443     1024   cuda


## Step 5 — Patch model.py to use alternative backbone

This cell shows how to integrate the new backbone into BOA's existing `model.py`.
We do a **minimal, non-destructive patch** — the original Mamba model is still accessible.

In [ ]:
with open('model.py', 'r') as f:
    original_model = f.read()

# Find where BoaConstrictor function starts and inject backbone switching
BACKBONE_INJECTION = '''
    # ── Backbone override via BOA_BACKBONE env var ──
    import os as _os
    _backbone = _os.environ.get('BOA_BACKBONE', 'mamba')
    if _backbone != 'mamba':
        from alternative_backbones import build_boa_model
        print(f'[BOA] Using alternative backbone: {_backbone}')
        return build_boa_model(backbone=_backbone, d_model=d_model,
                               num_layers=num_layers).to(device)
    # ── End backbone override ──
'''

# Inject right after the def BoaConstrictor line
patched = original_model.replace(
    'def BoaConstrictor(d_model=256, num_layers=4, vocab_size=256, device="cuda"):',
    'def BoaConstrictor(d_model=256, num_layers=4, vocab_size=256, device="cuda"):'
    + '\n' + BACKBONE_INJECTION
)

with open('model.py', 'w') as f:
    f.write(patched)

print('model.py patched successfully')
print('Verify injection:')
!grep -n "BOA_BACKBONE\|backbone override" model.py

model.py patched successfully
Verify injection:
8:    # ── Backbone override via BOA_BACKBONE env var ──
10:    _backbone = _os.environ.get('BOA_BACKBONE', 'mamba')
16:    # ── End backbone override ──
19:    BACKBONE = os.environ.get('BOA_BACKBONE', 'mamba').lower()


## Step 6 — Run BOA compression with LSTM backbone

This uses the existing `main.py` CLI with the CMS experiment config.

In [ ]:
import os

# Switch to LSTM backbone via environment variable
os.environ['BOA_BACKBONE'] = 'transformer'

# Check if the CMS experiment config exists
!ls experiments/

atlas_experiment  cfd_experiment  cms_experiment_lg
camel_experiment  cms_experiment  hepmc_experiment


In [ ]:
# View the CMS experiment config
!cat experiments/cms_experiment/cms_experiment.yaml

compression:
  chunks_count: 10000
  file_to_compress: CMS_DATA_float32.bin
dataloader:
  batch_size: 5
  seq_len: 10000
device: cuda
file_path: CMS_DATA_float32.bin
model:
  d_model: 256
  num_layers: 1
model_path: cms_experiment_final_model_fp32.pt
name: cms_experiment
precision: fp32
progress: true
splits:
- 0.8
- 0.1
- 0.1
training:
  epochs: 10
  lr: 5e-4


In [ ]:
cat alternative_backbones.py

"""
alternative_backbones.py
Drop-in replacements for the BytewiseMamba backbone used in BOA Constrictor.

USAGE
-----
In model.py (or wherever BoaConstrictor is assembled), swap the backbone:

    # Original:
    from model import BoaConstrictor          # uses BytewiseMamba internally

    # With this file, replace the backbone by passing backbone="lstm" or "transformer":
    from alternative_backbones import build_boa_model
    model = build_boa_model(backbone="lstm", d_model=256, num_layers=4)
    model = build_boa_model(backbone="transformer", d_model=256, num_layers=4)

The output interface is identical to BoaConstrictor:
    input  : (batch, seq_len)         -- integer byte values 0-255
    output : (batch, seq_len, 256)    -- logits over next-byte distribution

DESIGN NOTES
------------
- Both backbones share the same byte embedding + output projection.
- LSTM:        O(seq_len) per step, constant memory, CPU-friendly, stateful.
- Transformer: O(seq_len^2) attention, highly par

In [ ]:
# Run with the CMS pre-trained model (skip training, compress-only)
# --model-path loads pre-trained weights; --compress-only skips training
!python main.py \
    --config cms_experiment \
    --compress-only \
    --show-timings \
    --device cuda

cuda
Read 49920000 bytes from /content/boa-constrictor/boa-constrictor/boa-constrictor/boa-constrictor/boa-constrictor/boa-constrictor/boa-constrictor/experiments/cms_experiment/CMS_DATA_float32.bin in 0.05s
[BOA] Using alternative backbone: transformer
Loading pre-trained model from /content/boa-constrictor/boa-constrictor/boa-constrictor/boa-constrictor/boa-constrictor/boa-constrictor/boa-constrictor/experiments/cms_experiment/cms_experiment_final_model_fp32.pt
Skipping training (final model or explicit path provided).
Model loaded in 0.01s
Starting compression...
[compress] gpu_streams=5000 (chunk_len=4992, free=10.0GiB)
/content/boa-constrictor/boa-constrictor/boa-constrictor/boa-constrictor/boa-constrictor/boa-constrictor/boa-constrictor/boa.py:231: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it w

## Step 9 — Compare compression ratios

Record the output from each run and compare against the LZMA baseline.

In [ ]:
# Run LZMA and ZLIB baselines for comparison
!python main.py \
    --config cms_experiment \
    --comparison-baseline-only \
    --show-timings

cuda
Read 49920000 bytes from /content/boa-constrictor/boa-constrictor/boa-constrictor/boa-constrictor/boa-constrictor/boa-constrictor/experiments/cms_experiment/CMS_DATA_float32.bin in 0.03s
[BOA] Using alternative backbone: transformer

Baseline compression results:
  Original size: 49920000 bytes
  LZMA  -> size=15489644 bytes, ratio=3.22, time=51.073s, written=experiments/cms_experiment/cms_experiment.lzma
  ZLIB  -> size=19463909 bytes, ratio=2.56, time=23.335s, written=experiments/cms_experiment/cms_experiment.zlib

--comparison-baseline-only complete. Exiting.


In [ ]:
# Summary table — fill in results from the runs above
import pandas as pd

# Fill in from actual run outputs:
comparison = pd.DataFrame([
    {'Method': 'LZMA (baseline)',    'Compression Ratio': None, 'Throughput (MB/s)': None},
    {'Method': 'ZLIB (baseline)',    'Compression Ratio': None, 'Throughput (MB/s)': None},
    {'Method': 'BOA (Mamba)',        'Compression Ratio': None, 'Throughput (MB/s)': None},
    {'Method': 'BOA (LSTM)',         'Compression Ratio': None, 'Throughput (MB/s)': None},
    {'Method': 'BOA (Transformer)',  'Compression Ratio': None, 'Throughput (MB/s)': None},
])

print(comparison.to_string(index=False))

           Method Compression Ratio Throughput (MB/s)
  LZMA (baseline)              None              None
  ZLIB (baseline)              None              None
      BOA (Mamba)              None              None
       BOA (LSTM)              None              None
BOA (Transformer)              None              None


## Key Findings Template (for your logbook)

After running the cells above, fill in this section for your logbook and presentation:

| Metric | Mamba | LSTM | Transformer |
|--------|-------|------|-------------|
| Parameters | — | — | — |
| Throughput (inference) | — | — | — |
| Compression ratio vs LZMA | — | — | — |
| CUDA required? | Yes | No | No (but benefits) |
| Streaming support | Yes | Yes (stateful) | Limited |

### Challenges encountered
- **Mamba**: requires CUDA + `mamba-ssm` C++ extension; no CPU fallback
- **LSTM**: CPU-compatible, but slower inference; hidden state must be carefully passed between chunks for correct streaming
- **Transformer**: O(seq_len²) memory — seq_len=32768 (BOA default) is infeasible; must reduce to ~2048-4096 per chunk, which reduces context window and may hurt compression ratio
- Both alternatives need full retraining to match Mamba's compression ratio